# 02 - Extracción de texto de PDFs

Este notebook realiza la extracción de texto de los documentos PDF registrados en el corpus normativo ambiental.

Entrada principal:

- `data/metadata/corpus_normativo_ambiental.csv`
- PDFs ubicados en `data/raw/`

Salida generada:

- archivos `.txt` por documento en `data/processed/`
- reporte de extracción en `experiments/resultados/`
- versión actualizada del CSV con información de extracción en `data/metadata/`

Este paso prepara el corpus para la siguiente etapa: **chunking estructural**.


## 1. Importación de librerías

In [40]:
import os
import re
import sys
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

# Instalación / importación de PyMuPDF
try:
    import fitz  # PyMuPDF
except ModuleNotFoundError:
    print("PyMuPDF no está instalado. Instalando...")
    !{sys.executable} -m pip install pymupdf
    import fitz
    print("PyMuPDF instalado correctamente.")

print("Librerías importadas correctamente.")


Librerías importadas correctamente.


## 2. Definición de rutas del proyecto

El notebook puede ejecutarse desde la raíz del repositorio o desde la carpeta `notebooks/`.


In [30]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    ROOT_DIR = current_dir.parent
else:
    ROOT_DIR = current_dir

DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"
REPORTS_DIR = ROOT_DIR / "experiments" / "resultados"

CSV_PATH = METADATA_DIR / "corpus_normativo_ambiental.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"CSV_PATH: {CSV_PATH}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")


ROOT_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana
CSV_PATH: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\metadata\corpus_normativo_ambiental.csv
RAW_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\raw
PROCESSED_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\processed


## 3. Carga del corpus

In [31]:
df = pd.read_csv(CSV_PATH)

print(f"Documentos registrados en el corpus: {len(df)}")
df.head()


Documentos registrados en el corpus: 30


,id_documento,nombre_archivo_original,archivo_pdf,titulo_documento,tipo_norma,numero_norma,entidad_emisora,fecha_publicacion,tema_principal,subtema,fuente_oficial,url_fuente,texto_extraible,estado_vigencia,prioridad,observaciones
0,DOC-001,625.pdf,DOC-001_DS_004_2012_MINAM_IGAC.pdf,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,Minería y ambiente / IGAC,El Peruano,No determinado,Sí,No determinado,Alta,Norma relevante porque regula el IGAC aplicabl...
1,DOC-002,305-2017-MINAM.pdf,DOC-002_RM_305_2017_MINAM_CALIDAD_AIRE_GESTA.pdf,Aprueban los Lineamientos para el Fortalecimie...,Resolución Ministerial,Resolución Ministerial N.º 305-2017-MINAM,MINAM,2017-10-19,Calidad ambiental,ECA / Calidad del aire / GESTA,MINAM / El Peruano,No determinado,Parcial,No determinado,Alta,Norma relevante porque aprueba lineamientos pa...
2,DOC-003,656.pdf,DOC-003_RM_702_2008_MINSA_RESIDUOS_SEGREGADORE...,Aprueban la NTS N.º 073-2008-MINSA/DIGESA-V.01...,Resolución Ministerial,Resolución Ministerial N.º 702-2008/MINSA / NT...,MINSA / DIGESA,2008-10-09,Residuos sólidos,Residuos sólidos / Segregadores / Reaprovecham...,MINSA / DIGESA,http://www.minsa.gob.pe/portal/transparencia/n...,No,No determinado,Media,Documento oficial relevante porque establece u...
3,DOC-004,1466.pdf,DOC-004_LEY_28611_2005_CONGRESO_LEY_GENERAL_AM...,Ley General del Ambiente,Ley,Ley N.º 28611,Congreso de la República,2005-10-15,Gestión ambiental,SNGA / Gestión ambiental / Política ambiental,Congreso de la República / MINAM,https://www.leyes.congreso.gob.pe/documentos/l...,Sí,No determinado,Alta,Norma base del marco ambiental peruano. Establ...
4,DOC-005,1599.pdf,DOC-005_RM_554_2012_MINSA_RESIDUOS_ESTABLECIMI...,"Aprueban la NTS N.º 096-MINSA/DIGESA-V.01, Nor...",Resolución Ministerial,Resolución Ministerial N.º 554-2012/MINSA / NT...,MINSA / DIGESA,2012-07-03,Residuos sólidos,Residuos sólidos / Residuos hospitalarios / Ma...,MINSA / DIGESA,http://www.minsa.gob.pe/transparencia/dge_norm...,No,No determinado,Media,Documento oficial relevante porque regula la g...


## 4. Funciones de limpieza y extracción

La extracción se realizará con **PyMuPDF**.  
Se conservarán separadores por página para facilitar después el chunking y la trazabilidad.


In [32]:
def clean_text(text: str) -> str:
    """Limpieza básica del texto extraído."""
    if not isinstance(text, str):
        return ""

    # Normalizar saltos de línea
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Eliminar espacios excesivos por línea
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]

    # Eliminar múltiples líneas vacías consecutivas
    cleaned_lines = []
    previous_empty = False

    for line in lines:
        is_empty = line == ""
        if is_empty and previous_empty:
            continue
        cleaned_lines.append(line)
        previous_empty = is_empty

    return "\n".join(cleaned_lines).strip()


def extract_text_with_pymupdf(pdf_path: Path) -> dict:
    """Extrae texto de un PDF usando PyMuPDF y devuelve texto + métricas."""
    result = {
        "text": "",
        "pages": 0,
        "pages_with_text": 0,
        "empty_pages": 0,
        "characters": 0,
        "words": 0,
        "status": "ERROR",
        "error": None,
        "method": "PyMuPDF",
    }

    try:
        doc = fitz.open(pdf_path)
        result["pages"] = len(doc)

        page_texts = []

        for page_index, page in enumerate(doc, start=1):
            raw_text = page.get_text("text")
            cleaned_page_text = clean_text(raw_text)

            if cleaned_page_text:
                result["pages_with_text"] += 1
            else:
                result["empty_pages"] += 1

            page_texts.append(f"\n\n===== PÁGINA {page_index} =====\n\n{cleaned_page_text}")

        doc.close()

        full_text = clean_text("\n".join(page_texts))
        result["text"] = full_text
        result["characters"] = len(full_text)
        result["words"] = len(full_text.split())

        if result["characters"] == 0:
            result["status"] = "SIN_TEXTO"
        elif result["pages"] > 0 and result["pages_with_text"] / result["pages"] < 0.7:
            result["status"] = "PARCIAL"
        else:
            result["status"] = "OK"

    except Exception as e:
        result["error"] = str(e)
        result["status"] = "ERROR"

    return result


def infer_texto_extraible(status: str) -> str:
    """Convierte el estado técnico de extracción al valor usado en el CSV."""
    if status == "OK":
        return "Sí"
    if status == "PARCIAL":
        return "Parcial"
    if status == "SIN_TEXTO":
        return "No"
    return "No determinado"


## 5. Prueba rápida con el primer documento

Antes de procesar todo, se prueba con el primer PDF registrado en el corpus.


In [33]:
first_row = df.iloc[0]
first_pdf = RAW_DIR / first_row["archivo_pdf"]

print(f"Probando extracción con: {first_pdf.name}")

if not first_pdf.exists():
    print(f"No existe el archivo: {first_pdf}")
else:
    test_result = extract_text_with_pymupdf(first_pdf)
    print({
        "status": test_result["status"],
        "pages": test_result["pages"],
        "pages_with_text": test_result["pages_with_text"],
        "characters": test_result["characters"],
        "words": test_result["words"],
        "error": test_result["error"],
    })
    print("\nMuestra de texto extraído:\n")
    print(test_result["text"][:1500])


Probando extracción con: DOC-001_DS_004_2012_MINAM_IGAC.pdf
{'status': 'OK', 'pages': 7, 'pages_with_text': 7, 'characters': 42050, 'words': 5980, 'error': None}

Muestra de texto extraído:

===== PÁGINA 1 =====

NORMAS LEGALES
El Peruano
Lima, jueves 6 de setiembre de 2012
474028
Aprueban Disposiciones Complemen-
tarias para el Instrumento de Gestión
Ambiental Correctivo (IGAC), para la
Formalización de Actividades de Pe-
queña Minería y Minería Artesanal en
curso
DECRETO SUPREMO
Nº 004-2012-MINAM
EL PRESIDENTE DE LA REPÚBLICA
CONSIDERANDO:
Que, el derecho fundamental a gozar de un ambiente
equilibrado y adecuado al desarrollo de la vida, como
lo señala el inciso 22 del artículo 2º de la Constitución
Política del Perú, tiene relevancia para las generaciones
actuales como para las futuras, de acuerdo al Principio
de Sostenibilidad que establece el artículo V del Título
Preliminar de la Ley Nº 28611, Ley General del Ambiente;
Que, asimismo, el artículo 67º la Constitución Política
del P

## 6. Extracción de texto de todos los PDFs

Se procesa cada documento del CSV.  
Por cada PDF se genera un `.txt` en `data/processed/`.


In [34]:
extraction_results = []
updated_rows = []

for _, row in df.iterrows():
    id_documento = str(row["id_documento"])
    archivo_pdf = str(row["archivo_pdf"])
    pdf_path = RAW_DIR / archivo_pdf

    txt_filename = f"{id_documento}.txt"
    txt_path = PROCESSED_DIR / txt_filename

    if not pdf_path.exists():
        result_record = {
            "id_documento": id_documento,
            "archivo_pdf": archivo_pdf,
            "archivo_txt": txt_filename,
            "status": "PDF_NO_ENCONTRADO",
            "pages": 0,
            "pages_with_text": 0,
            "empty_pages": 0,
            "characters": 0,
            "words": 0,
            "method": "PyMuPDF",
            "error": f"No existe el archivo {pdf_path}",
        }
        extraction_results.append(result_record)

        updated_row = row.to_dict()
        updated_row["archivo_txt"] = txt_filename
        updated_row["estado_extraccion"] = "PDF_NO_ENCONTRADO"
        updated_row["caracteres_extraidos"] = 0
        updated_row["paginas_pdf"] = 0
        updated_row["texto_extraible_detectado"] = "No determinado"
        updated_rows.append(updated_row)
        continue

    extraction = extract_text_with_pymupdf(pdf_path)

    if extraction["status"] in {"OK", "PARCIAL"}:
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(extraction["text"])

    result_record = {
        "id_documento": id_documento,
        "archivo_pdf": archivo_pdf,
        "archivo_txt": txt_filename,
        "status": extraction["status"],
        "pages": extraction["pages"],
        "pages_with_text": extraction["pages_with_text"],
        "empty_pages": extraction["empty_pages"],
        "characters": extraction["characters"],
        "words": extraction["words"],
        "method": extraction["method"],
        "error": extraction["error"],
    }
    extraction_results.append(result_record)

    updated_row = row.to_dict()
    updated_row["archivo_txt"] = txt_filename
    updated_row["estado_extraccion"] = extraction["status"]
    updated_row["caracteres_extraidos"] = extraction["characters"]
    updated_row["paginas_pdf"] = extraction["pages"]
    updated_row["texto_extraible_detectado"] = infer_texto_extraible(extraction["status"])
    updated_rows.append(updated_row)

extraction_df = pd.DataFrame(extraction_results)
updated_df = pd.DataFrame(updated_rows)

extraction_df.head()


,id_documento,archivo_pdf,archivo_txt,status,pages,pages_with_text,empty_pages,characters,words,method,error
0,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,OK,7,7,0,42050,5980,PyMuPDF,None
1,DOC-002,DOC-002_RM_305_2017_MINAM_CALIDAD_AIRE_GESTA.pdf,DOC-002.txt,OK,13,13,0,29068,4309,PyMuPDF,None
2,DOC-003,DOC-003_RM_702_2008_MINSA_RESIDUOS_SEGREGADORE...,DOC-003.txt,PARCIAL,11,0,11,242,44,PyMuPDF,None
3,DOC-004,DOC-004_LEY_28611_2005_CONGRESO_LEY_GENERAL_AM...,DOC-004.txt,OK,46,46,0,122923,18282,PyMuPDF,None
4,DOC-005,DOC-005_RM_554_2012_MINSA_RESIDUOS_ESTABLECIMI...,DOC-005.txt,PARCIAL,60,0,60,1369,240,PyMuPDF,None


## 7. Resumen de extracción

In [35]:
print("Resumen por estado de extracción:")
display(extraction_df["status"].value_counts().reset_index().rename(columns={"index": "estado", "status": "cantidad"}))

print("Documentos con menor cantidad de caracteres extraídos:")
display(extraction_df.sort_values("characters").head(10))

print("Documentos con mayor cantidad de caracteres extraídos:")
display(extraction_df.sort_values("characters", ascending=False).head(10))


Resumen por estado de extracción:


,cantidad,count
0,OK,26
1,PARCIAL,4


Documentos con menor cantidad de caracteres extraídos:


,id_documento,archivo_pdf,archivo_txt,status,pages,pages_with_text,empty_pages,characters,words,method,error
26,DOC-027,DOC-027_RM_209_2009_MINAM_PLANES_CONTINGENCIA_...,DOC-027.txt,PARCIAL,2,0,2,42,8,PyMuPDF,None
2,DOC-003,DOC-003_RM_702_2008_MINSA_RESIDUOS_SEGREGADORE...,DOC-003.txt,PARCIAL,11,0,11,242,44,PyMuPDF,None
4,DOC-005,DOC-005_RM_554_2012_MINSA_RESIDUOS_ESTABLECIMI...,DOC-005.txt,PARCIAL,60,0,60,1369,240,PyMuPDF,None
23,DOC-024,DOC-024_PROTOCOLO_2019_MINAM_MONITOREO_CALIDAD...,DOC-024.txt,PARCIAL,102,0,102,2338,408,PyMuPDF,None
29,DOC-030,DOC-030_ORD_REG_005_2018_GORE_PUNO_FISCALIZACI...,DOC-030.txt,OK,1,1,0,6849,1025,PyMuPDF,None
10,DOC-011,DOC-011_DS_003_2008_MINAM_ECA_AIRE.pdf,DOC-011.txt,OK,4,4,0,7061,1117,PyMuPDF,None
14,DOC-015,DOC-015_DS_012_2018_MINAM_CALIDAD_AIRE_EMISION...,DOC-015.txt,OK,2,2,0,11460,1725,PyMuPDF,None
12,DOC-013,DOC-013_FE_ERRATAS_DS_009_2015_MINAM_CALIDAD_A...,DOC-013.txt,OK,2,2,0,11650,1775,PyMuPDF,None
25,DOC-026,DOC-026_RM_181_2016_MINAM_INCA_INFO_AIRE.pdf,DOC-026.txt,OK,6,6,0,12421,1963,PyMuPDF,None
11,DOC-012,DOC-012_DS_009_2015_MINAM_CALIDAD_AIRE_DIESEL_...,DOC-012.txt,OK,2,2,0,13048,2095,PyMuPDF,None


Documentos con mayor cantidad de caracteres extraídos:


,id_documento,archivo_pdf,archivo_txt,status,pages,pages_with_text,empty_pages,characters,words,method,error
22,DOC-023,DOC-023_COMPENDIO_MINAM_LEY_GENERAL_AMBIENTE_S...,DOC-023.txt,OK,168,168,0,315695,46942,PyMuPDF,None
9,DOC-010,DOC-010_DS_005_2026_MINAM_SUSTANCIAS_QUIMICAS.pdf,DOC-010.txt,OK,24,24,0,157935,24690,PyMuPDF,None
13,DOC-014,DOC-014_DS_018_2025_MINAM_MONITOREO_SEDIMENTOS...,DOC-014.txt,OK,36,36,0,154297,23703,PyMuPDF,None
3,DOC-004,DOC-004_LEY_28611_2005_CONGRESO_LEY_GENERAL_AM...,DOC-004.txt,OK,46,46,0,122923,18282,PyMuPDF,None
20,DOC-021,DOC-021_INFORME_RS_189_2012_PCM_EJES_GESTION_A...,DOC-021.txt,OK,32,32,0,90893,13949,PyMuPDF,None
8,DOC-009,DOC-009_DS_006_2026_MTC_PROTECCION_AMBIENTAL_T...,DOC-009.txt,OK,10,10,0,75275,11353,PyMuPDF,None
16,DOC-017,DOC-017_DS_004_2017_MINAM_ECA_AGUA.pdf,DOC-017.txt,OK,10,10,0,44067,7040,PyMuPDF,None
0,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,OK,7,7,0,42050,5980,PyMuPDF,None
24,DOC-025,DOC-025_RES_SBS_1928_2015_SBS_RIESGO_SOCIAL_AM...,DOC-025.txt,OK,5,5,0,34680,5460,PyMuPDF,None
1,DOC-002,DOC-002_RM_305_2017_MINAM_CALIDAD_AIRE_GESTA.pdf,DOC-002.txt,OK,13,13,0,29068,4309,PyMuPDF,None


## 8. Revisión de documentos problemáticos

Se listan documentos que no pudieron extraerse correctamente o que tienen extracción parcial.


In [36]:
problematic = extraction_df[extraction_df["status"].isin(["PDF_NO_ENCONTRADO", "SIN_TEXTO", "PARCIAL", "ERROR"])]

if problematic.empty:
    print("No se detectaron documentos problemáticos.")
else:
    print("Documentos a revisar:")
    display(problematic)


Documentos a revisar:


,id_documento,archivo_pdf,archivo_txt,status,pages,pages_with_text,empty_pages,characters,words,method,error
2,DOC-003,DOC-003_RM_702_2008_MINSA_RESIDUOS_SEGREGADORE...,DOC-003.txt,PARCIAL,11,0,11,242,44,PyMuPDF,None
4,DOC-005,DOC-005_RM_554_2012_MINSA_RESIDUOS_ESTABLECIMI...,DOC-005.txt,PARCIAL,60,0,60,1369,240,PyMuPDF,None
23,DOC-024,DOC-024_PROTOCOLO_2019_MINAM_MONITOREO_CALIDAD...,DOC-024.txt,PARCIAL,102,0,102,2338,408,PyMuPDF,None
26,DOC-027,DOC-027_RM_209_2009_MINAM_PLANES_CONTINGENCIA_...,DOC-027.txt,PARCIAL,2,0,2,42,8,PyMuPDF,None


## 9. Guardado de reportes

Se guardan:

- reporte de extracción en CSV;
- corpus actualizado con información de extracción;
- reporte JSON resumen.


In [37]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

extraction_report_path = REPORTS_DIR / f"reporte_extraccion_texto_{timestamp}.csv"
updated_corpus_path = METADATA_DIR / "corpus_normativo_ambiental_con_extraccion.csv"
summary_json_path = REPORTS_DIR / f"resumen_extraccion_texto_{timestamp}.json"

extraction_df.to_csv(extraction_report_path, index=False, encoding="utf-8-sig")
updated_df.to_csv(updated_corpus_path, index=False, encoding="utf-8-sig")

summary = {
    "fecha_extraccion": datetime.now().isoformat(timespec="seconds"),
    "total_documentos": int(len(extraction_df)),
    "total_ok": int((extraction_df["status"] == "OK").sum()),
    "total_parcial": int((extraction_df["status"] == "PARCIAL").sum()),
    "total_sin_texto": int((extraction_df["status"] == "SIN_TEXTO").sum()),
    "total_error": int((extraction_df["status"] == "ERROR").sum()),
    "total_pdf_no_encontrado": int((extraction_df["status"] == "PDF_NO_ENCONTRADO").sum()),
    "total_caracteres": int(extraction_df["characters"].sum()),
    "total_palabras": int(extraction_df["words"].sum()),
}

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=4)

print(f"Reporte de extracción guardado en: {extraction_report_path}")
print(f"Corpus actualizado guardado en: {updated_corpus_path}")
print(f"Resumen JSON guardado en: {summary_json_path}")


Reporte de extracción guardado en: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\experiments\resultados\reporte_extraccion_texto_20260519_212536.csv
Corpus actualizado guardado en: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\metadata\corpus_normativo_ambiental_con_extraccion.csv
Resumen JSON guardado en: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\experiments\resultados\resumen_extraccion_texto_20260519_212536.json


## 10. Verificación de archivos `.txt` generados

In [38]:
txt_files = sorted(PROCESSED_DIR.glob("*.txt"))

print(f"Archivos .txt generados: {len(txt_files)}")

for txt_file in txt_files[:10]:
    print(f"- {txt_file.name}")

if len(txt_files) > 10:
    print("...")


Archivos .txt generados: 30
- DOC-001.txt
- DOC-002.txt
- DOC-003.txt
- DOC-004.txt
- DOC-005.txt
- DOC-006.txt
- DOC-007.txt
- DOC-008.txt
- DOC-009.txt
- DOC-010.txt
...


## 11. Resultado final

Si la mayoría de documentos quedó con estado `OK`, el corpus está listo para pasar al **Notebook 03 - Chunking estructural**.


In [39]:
if (extraction_df["status"] == "OK").sum() > 0:
    print("Extracción completada. Revisa los reportes y los documentos problemáticos antes de continuar.")
else:
    print("No se logró extraer texto correctamente. Revisa los PDFs o considera OCR.")

display(extraction_df)


Extracción completada. Revisa los reportes y los documentos problemáticos antes de continuar.


,id_documento,archivo_pdf,archivo_txt,status,pages,pages_with_text,empty_pages,characters,words,method,error
0,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,OK,7,7,0,42050,5980,PyMuPDF,None
1,DOC-002,DOC-002_RM_305_2017_MINAM_CALIDAD_AIRE_GESTA.pdf,DOC-002.txt,OK,13,13,0,29068,4309,PyMuPDF,None
2,DOC-003,DOC-003_RM_702_2008_MINSA_RESIDUOS_SEGREGADORE...,DOC-003.txt,PARCIAL,11,0,11,242,44,PyMuPDF,None
3,DOC-004,DOC-004_LEY_28611_2005_CONGRESO_LEY_GENERAL_AM...,DOC-004.txt,OK,46,46,0,122923,18282,PyMuPDF,None
4,DOC-005,DOC-005_RM_554_2012_MINSA_RESIDUOS_ESTABLECIMI...,DOC-005.txt,PARCIAL,60,0,60,1369,240,PyMuPDF,None
5,DOC-006,DOC-006_RM_415_2012_MINSA_AMBIENTES_LIBRES_HUM...,DOC-006.txt,OK,8,8,0,18092,2932,PyMuPDF,None
6,DOC-007,DOC-007_DS_011_2023_MINAM_ECA_AIRE_PM10_METALE...,DOC-007.txt,OK,3,3,0,19676,3062,PyMuPDF,None
7,DOC-008,DOC-008_DS_017_2025_MINAM_GESTION_CALIDAD_AIRE...,DOC-008.txt,OK,2,2,0,14453,2173,PyMuPDF,None
8,DOC-009,DOC-009_DS_006_2026_MTC_PROTECCION_AMBIENTAL_T...,DOC-009.txt,OK,10,10,0,75275,11353,PyMuPDF,None
9,DOC-010,DOC-010_DS_005_2026_MINAM_SUSTANCIAS_QUIMICAS.pdf,DOC-010.txt,OK,24,24,0,157935,24690,PyMuPDF,None
